# Part 2 Test-Time Scaling

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [1]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# !pip install vllm==0.20.0
from vllm import LLM, SamplingParams

ModuleNotFoundError: No module named 'vllm'

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Thinking
DATASET_NAME = "OpenRLHF/aime-2024"
MAX_NEW_TOKENS = 8192 # Reduced from 16384 to mitigate memory issues


## Loading the model and the data

In [ ]:
import os
import sys

# Using vLLM instead for faster inference
dtype = "float16" if torch.cuda.is_available() else "float32"

# Redirect stdout and stderr to os.devnull temporarily to avoid fileno() error
original_stdout = sys.stdout
original_stderr = sys.stderr
sys.stdout = open(os.devnull, 'w')
sys.stderr = open(os.devnull, 'w')

try:
    # Decreasing max_model_len to fit available GPU memory
    llm       = LLM(model=MODEL_NAME, dtype=dtype, max_model_len=MAX_NEW_TOKENS, hf_token=HF_TOKEN)
finally:
    sys.stdout.close()
    sys.stderr.close()
    sys.stdout = original_stdout
    sys.stderr = original_stderr

tokenizer = llm.get_tokenizer()

dataset = load_dataset(DATASET_NAME, split="train", token=HF_TOKEN)

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32,
# )

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# dataset = load_dataset(DATASET_NAME, split="train")


## Evaluation helpers

In [ ]:
def extract_thinking_trace(text):
    # complete thinking
    match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # truncated due to max tokens
    match = re.search(r'<think>(.*)', text, re.DOTALL)
    return match.group(1).strip() if match else ""

def think_end_ids(tokenizer):
    for tag in ["<|/thinking|>", "</think>"]:
        ids = tokenizer.encode(tag, add_special_tokens=False)
        if ids:
            return ids, tag

def make_prompt_think(text, tokenizer):
    msgs = [
        {"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
        {"role": "user",   "content": text},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)

def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()


def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text


    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None

    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None


## 2.1 Warm-Up

You can also explore using vLLM to speed up inference!

In [ ]:
# ## DEPRECIATED: too slow!
# ENABLE_THINKING = True  # Ensure thinking is enabled for this analysis
# ANSWER_MODE = "flexible_extract"

# model.to(device)
# model.eval()

# records = []

# for i, example in enumerate(dataset):
#     problem = example["prompt"][0]["content"]
#     gold_answer = int(example["label"])

#     messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
#                 {"role": "user", "content": problem}]
#     prompt = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=ENABLE_THINKING,
#     )

#     inputs = tokenizer(prompt, return_tensors="pt").to(device)

#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=MAX_NEW_TOKENS,
#             do_sample=False,
#             temperature=None,
#             top_p=None,
#         )

#     # Decode only the newly generated tokens
#     generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
#     model_output = tokenizer.decode(generated_ids, skip_special_tokens=False)

#     # Debug: print raw tags on first example to confirm tag format
#     if i == 0:
#         print("RAW OUTPUT SNIPPET:", repr(model_output[:300]))

#     extracted = extract_answer(model_output, mode=ANSWER_MODE)
#     if extracted is not None:
#         correct = extracted == gold_answer
#     else:
#         correct = False

#     # Extract thinking trace and calculate its token length
#     extracted_thinking = extract_thinking_trace(model_output)
#     thinking_tokens = tokenizer.encode(extracted_thinking, add_special_tokens=False)
#     thinking_length = len(thinking_tokens)

#     records.append({
#         "problem": problem,
#         "gold_answer": gold_answer,
#         "model_output": model_output,
#         "extracted_answer": extracted,
#         "correct": correct,
#         "thinking_length": thinking_length, # Add thinking length to records
#     })

#     print(f"[{i+1}/{len(dataset)}] gold={gold_answer} pred={extracted} correct={correct} thinking_len={thinking_length}")

# results_df = pd.DataFrame(records)

# print("\nDistribution of Thinking Lengths (in tokens):")
# print(results_df["thinking_length"].describe())

# results_df

In [ ]:
#vLLM instead!
ANSWER_MODE = 'flexible_extract'
prompts, gold_answers, problems = [], [], []
for example in dataset:
    problem = example["prompt"][0]["content"]
    problems.append(problem)
    gold_answers.append(int(example["label"]))
    prompts.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
         {"role": "user",   "content": problem}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True,
    ))

sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, detokenize=True, skip_special_tokens=False)
outputs  = llm.generate(prompts, sampling_params=sampling)

print("RAW OUTPUT SNIPPET:", repr(outputs[0].outputs[0].text[:300]))

records = []
for i, (out, gold, problem) in enumerate(zip(outputs, gold_answers, problems)):
    model_output = out.outputs[0].text
    extracted    = extract_answer(model_output, mode=ANSWER_MODE)
    correct      = extracted == gold if extracted is not None else False
    thinking     = extract_thinking_trace(model_output)
    think_len    = len(tokenizer.encode(thinking, add_special_tokens=False))

    records.append(dict(problem=problem, gold_answer=gold, model_output=model_output,
                        extracted_answer=extracted, correct=correct, thinking_length=think_len))
    print(f"[{i+1}/{len(dataset)}] gold={gold} pred={extracted} correct={correct} thinking_len={think_len}")

results_df = pd.DataFrame(records)
print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df["thinking_length"].describe())

prompts_no_thinking = []
for i, example in enumerate(dataset):
    prompts_no_thinking.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
         {"role": "user",   "content": problems[i]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    ))

sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, detokenize=True, skip_special_tokens=False)
outputs_no_thinking  = llm.generate(prompts_no_thinking, sampling_params=sampling)

print("RAW OUTPUT SNIPPET:", repr(outputs_no_thinking[0].outputs[0].text[:300]))

records_no_thinking = []
for i, (out, gold, problem) in enumerate(zip(outputs_no_thinking, gold_answers, problems)):
    model_output = out.outputs[0].text
    extracted    = extract_answer(model_output, mode=ANSWER_MODE)
    correct      = extracted == gold if extracted is not None else False
    thinking     = extract_thinking_trace(model_output)
    think_len    = len(tokenizer.encode(thinking, add_special_tokens=False))

    records_no_thinking.append(dict(problem=problem, gold_answer=gold, model_output=model_output,
                        extracted_answer=extracted, correct=correct, thinking_length=think_len))
    print(f"[{i+1}/{len(dataset)}] gold={gold} pred={extracted} correct={correct} thinking_len={think_len}")

results_df_no_thinking = pd.DataFrame(records_no_thinking)
print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df_no_thinking["thinking_length"].describe())

In [ ]:
print(f'Thinking-enabled accuracy: {results_df["correct"].mean()}')
print(f'No-thinking accuracy: {results_df_no_thinking["correct"].mean()}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Histogram for Thinking-enabled mode
sns.histplot(results_df['thinking_length'], bins=50, color='skyblue', label='Thinking Enabled', kde=True)

# Histogram for No-thinking mode (will likely be all 0s)
sns.histplot(results_df_no_thinking['thinking_length'], bins=50, color='salmon', label='No Thinking', kde=True)

plt.title('Distribution of Thinking Lengths (in Tokens) for Both Inference Modes')
plt.xlabel('Thinking Length (Tokens)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 2.2 Scaling Experiments

In [ ]:

SEQ_BUDGETS    = [1024, 2048, 4096, 8192, 16384, 32000]
PAR_M_VALUES   = [1, 2, 4, 8, 16, 32]
PAR_BUDGET     = 4000
MAX_ANS_TOKENS = 512
N_RUNS         = 3          # re-runs for parallel error bars
T, TOP_K, TOP_P = 0.6, 50, 0.95

In [ ]:
def run_one(llm, tokenizer, problem, budget, strategy="wait",
            do_sample=False, temperature=0.6, top_k=50, top_p=0.95):
    end_ids, end_tag = think_end_ids(tokenizer)
    prompt   = make_prompt_think(problem, tokenizer)
    sampling = SamplingParams(
        temperature      = temperature if do_sample else 0.0,
        top_k            = top_k  if do_sample else -1,
        top_p            = top_p  if do_sample else 1.0,
        max_tokens       = 1,
        detokenize       = False,
    )

    token_ids = tokenizer.encode(prompt)

    # ── thinking loop ─────────────────────────────────────────────────────────
    while True:
        think_len = len(token_ids) - len(tokenizer.encode(prompt))
        out       = llm.generate(prompt_token_ids=[token_ids], sampling_params=sampling)
        next_tok  = out[0].outputs[0].token_ids[0]

        if think_len >= budget:
            token_ids += end_ids
            break
        elif next_tok in end_ids and strategy == "wait":
            wait_id = tokenizer.encode("Wait", add_special_tokens=False)[0]
            token_ids.append(wait_id)
        else:
            token_ids.append(next_tok)
            if next_tok in end_ids:
                break

    # ── generate final answer ─────────────────────────────────────────────────
    ans_sampling = SamplingParams(
        temperature = temperature if do_sample else 0.0,
        top_k       = top_k  if do_sample else -1,
        top_p       = top_p  if do_sample else 1.0,
        max_tokens  = MAX_ANS_TOKENS,
    )
    out      = llm.generate(prompt_token_ids=[token_ids], sampling_params=ans_sampling)
    decoded  = out[0].outputs[0].text
    full_out = tokenizer.decode(token_ids) + decoded

    thinking  = extract_thinking_trace(full_out)
    think_tok = len(tokenizer.encode(thinking, add_special_tokens=False))
    total_tok = len(token_ids) - len(tokenizer.encode(prompt)) + len(out[0].outputs[0].token_ids)
    return full_out, think_tok, total_tok
